# 01 - Generar entregables del challenge

Notebook operativo para reconstruir el proyecto desde cero: ingesta, dataset `comercial`, dashboard HTML y validaciones.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)


c:\Users\chris\Downloads\ia-challenge


## 1. Validar configuracion

Valida que existen credenciales en `.env` sin mostrar la contrasena.


In [2]:
from src.utils.config_loader import load_db_config, get_jdbc_url

db_config = load_db_config(require_credentials=True)
print("Host:", db_config["host"])
print("Database:", db_config["database"])
print("User configured:", bool(db_config["user"]))
print("Password configured:", bool(db_config["password"]))
print("JDBC URL:", get_jdbc_url(db_config))


Host: www.bigdataybi.com
Database: fake
User configured: True
Password configured: True
JDBC URL: jdbc:mysql://www.bigdataybi.com:3306/fake?useSSL=false&allowPublicKeyRetrieval=true


## 2. Crear SparkSession


In [3]:
from src.utils.spark_session import create_spark_session

spark = create_spark_session("challenge_entregables")
print("Spark version:", spark.version)


Spark version: 3.5.1


## 3. Ingerir tablas fuente

Ingiere `customers`, `products`, `sales` y `shops` hacia `data/raw/`.


In [4]:
from src.ingest.extract_tables import ingest_tables

raw_counts = ingest_tables(spark=spark)
raw_counts


2026-06-23 09:44:27,515 [INFO] src.ingest.extract_tables - Reading source table customers from www.bigdataybi.com:3306/fake
2026-06-23 09:44:31,254 [INFO] src.ingest.extract_tables - Writing raw table customers to C:\Users\chris\Downloads\ia-challenge\data\raw\customers
2026-06-23 09:44:32,338 [INFO] src.ingest.extract_tables - Ingested 10 rows from customers
2026-06-23 09:44:32,338 [INFO] src.ingest.extract_tables - Reading source table products from www.bigdataybi.com:3306/fake
2026-06-23 09:44:33,321 [INFO] src.ingest.extract_tables - Writing raw table products to C:\Users\chris\Downloads\ia-challenge\data\raw\products
2026-06-23 09:44:33,872 [INFO] src.ingest.extract_tables - Ingested 5 rows from products
2026-06-23 09:44:33,888 [INFO] src.ingest.extract_tables - Reading source table sales from www.bigdataybi.com:3306/fake
2026-06-23 09:44:34,805 [INFO] src.ingest.extract_tables - Writing raw table sales to C:\Users\chris\Downloads\ia-challenge\data\raw\sales
2026-06-23 09:44:35,34

{'customers': 10, 'products': 5, 'sales': 100, 'shops': 3}

## 4. Construir dataset comercial

Genera `data/analytics/comercial/` en formato Parquet.


In [5]:
from src.transform.build_comercial import build_comercial_table

comercial_rows = build_comercial_table(spark=spark)
print("Rows comercial:", comercial_rows)


2026-06-23 09:46:21,240 [INFO] src.transform.build_comercial - Writing comercial dataset to C:\Users\chris\Downloads\ia-challenge\data\analytics\comercial
2026-06-23 09:46:22,264 [INFO] src.transform.build_comercial - Comercial dataset written with 100 rows


Rows comercial: 100


## 5. Validar Parquet comercial


In [6]:
comercial_path = PROJECT_ROOT / "data" / "analytics" / "comercial"
comercial_df = spark.read.parquet(str(comercial_path))
print("Rows:", comercial_df.count())
comercial_df.select("invoice_id", "sale_date", "shop_name", "product_name", "customer_status", "subtotal").show(10, truncate=False)


Rows: 100
+----------+----------+------------------------+-----------------------+---------------+--------+
|invoice_id|sale_date |shop_name               |product_name           |customer_status|subtotal|
+----------+----------+------------------------+-----------------------+---------------+--------+
|76        |2025-06-24|Sucursal Guayaquil Norte|Cloro Concentrado      |activo         |5.41    |
|26        |2025-05-30|Sucursal Cuenca Centro  |Limpiador Desinfectante|activo         |73.0    |
|1         |2025-06-12|NULL                    |Limpiador Desinfectante|activo         |3.16    |
|77        |2025-06-03|Sucursal Cuenca Centro  |Limpiavidrios          |activo         |27.78   |
|27        |2025-05-31|Sucursal Cuenca Centro  |Limpiavidrios          |activo         |54.6    |
|2         |2025-06-11|Sucursal Guayaquil Norte|Limpiavidrios          |inactivo       |8.9     |
|3         |2025-06-04|NULL                    |Cloro Concentrado      |activo         |24.68   |
|78       

## 6. Generar dashboard HTML

Genera `dashboard/index.html` con KPI y 4 graficos.


In [7]:
from src.export.dashboard_data import generate_dashboard

payload = generate_dashboard()
print("Rows dashboard:", payload["metadata"]["row_count"])
print("Total vendido:", payload["kpis"]["total_sold"])
print("Archivo:", PROJECT_ROOT / "dashboard" / "index.html")


Rows dashboard: 100
Total vendido: 3028.54
Archivo: c:\Users\chris\Downloads\ia-challenge\dashboard\index.html


## 7. Validar componentes del dashboard


In [8]:
html = (PROJECT_ROOT / "dashboard" / "index.html").read_text(encoding="utf-8")
required = ["Total vendido", "dailySalesChart", "shopPieChart", "productBarChart", "customerStatusChart"]
for marker in required:
    print(marker, marker in html)


Total vendido True
dailySalesChart True
shopPieChart True
productBarChart True
customerStatusChart True


## 8. Cerrar Spark


In [9]:
spark.stop()
print("Spark stopped")


Spark stopped
